# Ajnyana — Training

Pre-train nanoGPT (~9M params) on 122M Sundanese tokens.

**Run on:** Kaggle T4 GPU (~17 min / 10K steps)

**Kaggle datasets needed:**
- `ajnyana-tokenizer` — `vocab.json` + `merges.txt`
- `ajnyana-corpus` — `train.txt` + `val.txt`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import os
import time
import json
from dataclasses import dataclass
from tokenizers import ByteLevelBPETokenizer

# ── Paths ──────────────────────────────────────────────────
ON_KAGGLE = os.path.exists('/kaggle')
if ON_KAGGLE:
    TOKENIZER_DIR = '/kaggle/input/ajnyana-tokenizer'
    CORPUS_TRAIN  = '/kaggle/input/ajnyana-corpus/train.txt'
    CORPUS_VAL    = '/kaggle/input/ajnyana-corpus/val.txt'
    OUT_DIR       = '/kaggle/working'
else:
    TOKENIZER_DIR = '../tokenizer'
    CORPUS_TRAIN  = '../data/processed/train.txt'
    CORPUS_VAL    = '../data/processed/val.txt'
    OUT_DIR       = '../data/processed'

TRAIN_BIN = os.path.join(OUT_DIR, 'train.bin')
VAL_BIN   = os.path.join(OUT_DIR, 'val.bin')
CKPT_DIR  = os.path.join(OUT_DIR, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device:   {device}')
print(f'Kaggle:   {ON_KAGGLE}')
if device == 'cuda':
    print(f'GPU:      {torch.cuda.get_device_name(0)}')
    print(f'VRAM:     {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 1. Model Architecture

In [ ]:
@dataclass
class GPTConfig:
    vocab_size : int   = 16_000
    block_size : int   = 512
    n_layer    : int   = 6
    n_head     : int   = 4
    n_embd     : int   = 256
    dropout    : float = 0.1
    bias       : bool  = False


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.n_embd = cfg.n_embd
        self.c_attn     = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=cfg.bias)
        self.c_proj     = nn.Linear(cfg.n_embd, cfg.n_embd,     bias=cfg.bias)
        self.attn_drop  = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.register_buffer('bias', torch.tril(torch.ones(cfg.block_size, cfg.block_size))
                             .view(1, 1, cfg.block_size, cfg.block_size))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        hd = C // self.n_head
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(hd))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = self.attn_drop(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc   = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=cfg.bias)
        self.c_proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.drop   = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.c_proj(F.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1  = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.attn = CausalSelfAttention(cfg)
        self.ln2  = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.mlp  = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(cfg.vocab_size, cfg.n_embd),
            wpe  = nn.Embedding(cfg.block_size, cfg.n_embd),
            drop = nn.Dropout(cfg.dropout),
            h    = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)]),
            ln_f = nn.LayerNorm(cfg.n_embd, bias=cfg.bias),
        ))
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        logits = self.lm_head(self.transformer.ln_f(x))
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_c = idx if idx.size(1) <= self.cfg.block_size else idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_c)
            logits = logits[:, -1, :] / temperature
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            idx = torch.cat((idx, torch.multinomial(F.softmax(logits, -1), 1)), dim=1)
        return idx

print('Model OK')

## 2. Tokenize Corpus → .bin

In [ ]:
def tokenize_to_bin(txt_path, bin_path, tokenizer, desc=''):
    if os.path.exists(bin_path):
        data = np.fromfile(bin_path, dtype=np.uint16)
        print(f'{desc} .bin exists — {len(data):,} tokens')
        return data

    print(f'Tokenizing {desc}...')
    with open(txt_path, 'r', encoding='utf-8') as f:
        docs = [d.strip() for d in f.read().split('\n\n') if d.strip()]

    EOT = tokenizer.token_to_id('<|endoftext|>')
    all_ids = []
    for i, doc in enumerate(docs):
        all_ids.extend(tokenizer.encode(doc).ids + [EOT])
        if (i + 1) % 10_000 == 0:
            print(f'  {i+1:,}/{len(docs):,} docs | {len(all_ids):,} tokens')

    arr = np.array(all_ids, dtype=np.uint16)
    arr.tofile(bin_path)
    print(f'Saved {len(arr):,} tokens → {bin_path} ({arr.nbytes/1e6:.1f} MB)')
    return arr


tokenizer = ByteLevelBPETokenizer(
    vocab   = f'{TOKENIZER_DIR}/vocab.json',
    merges  = f'{TOKENIZER_DIR}/merges.txt',
)
print(f'Tokenizer vocab: {tokenizer.get_vocab_size():,}')

train_data = tokenize_to_bin(CORPUS_TRAIN, TRAIN_BIN, tokenizer, 'train')
val_data   = tokenize_to_bin(CORPUS_VAL,   VAL_BIN,   tokenizer, 'val')

print(f'\nTrain tokens: {len(train_data):,}')
print(f'Val tokens:   {len(val_data):,}')

## 3. Training Config

In [ ]:
# ── Hyperparams ────────────────────────────────────────────
BATCH_SIZE     = 64
BLOCK_SIZE     = 512
MAX_ITERS      = 10_000
EVAL_INTERVAL  = 500
EVAL_ITERS     = 50
SAVE_INTERVAL  = 1_000
RESUME_FROM    = None   # path to checkpoint .pt, or None for fresh

# LR schedule (cosine with warmup)
LR_MAX         = 1e-3
LR_MIN         = 1e-4
WARMUP_ITERS   = 500

# Optimizer
WEIGHT_DECAY   = 0.1
BETA1, BETA2   = 0.9, 0.95
GRAD_CLIP      = 1.0

tokens_per_step = BATCH_SIZE * BLOCK_SIZE
total_tokens    = tokens_per_step * MAX_ITERS
epochs_equiv    = total_tokens / len(train_data)

print(f'Tokens/step:   {tokens_per_step:,}')
print(f'Total tokens:  {total_tokens/1e6:.1f}M')
print(f'Epochs equiv:  {epochs_equiv:.2f}x over train corpus')
print(f'Eval every:    {EVAL_INTERVAL} steps')
print(f'Save every:    {SAVE_INTERVAL} steps')

## 4. Data Loader

In [ ]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([torch.from_numpy(data[i  :i+BLOCK_SIZE  ].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+BLOCK_SIZE+1].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model):
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = []
        for _ in range(EVAL_ITERS):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

print('Data loader OK')

## 5. Model + Optimizer

In [ ]:
cfg   = GPTConfig()
model = GPT(cfg).to(device)

# Optimizer — separate weight decay for 2D params only
decay     = [p for p in model.parameters() if p.requires_grad and p.dim() >= 2]
no_decay  = [p for p in model.parameters() if p.requires_grad and p.dim() <  2]
optimizer = torch.optim.AdamW(
    [{'params': decay, 'weight_decay': WEIGHT_DECAY},
     {'params': no_decay, 'weight_decay': 0.0}],
    lr=LR_MAX, betas=(BETA1, BETA2)
)

start_iter = 0
if RESUME_FROM and os.path.exists(RESUME_FROM):
    ckpt = torch.load(RESUME_FROM, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_iter = ckpt['iter']
    print(f'Resumed from step {start_iter}')

total_params = sum(p.numel() for p in set(model.parameters()))
print(f'Model: {total_params/1e6:.2f}M params')
print(f'Start iter: {start_iter}')

## 6. LR Schedule (cosine + warmup)

In [ ]:
def get_lr(it):
    if it < WARMUP_ITERS:
        return LR_MAX * it / WARMUP_ITERS
    if it > MAX_ITERS:
        return LR_MIN
    decay_ratio = (it - WARMUP_ITERS) / (MAX_ITERS - WARMUP_ITERS)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return LR_MIN + coeff * (LR_MAX - LR_MIN)

print('LR schedule:')
for it in [0, 250, 500, 1000, 5000, 10000]:
    print(f'  step {it:>6}: lr = {get_lr(it):.2e}')

## 7. Training Loop

In [ ]:
scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))
log    = []

model.train()
t0 = time.time()

for it in range(start_iter, MAX_ITERS + 1):

    # ── LR update ──────────────────────────────────────────
    lr = get_lr(it)
    for pg in optimizer.param_groups:
        pg['lr'] = lr

    # ── Eval ───────────────────────────────────────────────
    if it % EVAL_INTERVAL == 0:
        losses = estimate_loss(model)
        elapsed = time.time() - t0
        log.append({'iter': it, 'train': losses['train'], 'val': losses['val'], 'lr': lr})
        print(f'step {it:>6} | train {losses["train"]:.4f} | val {losses["val"]:.4f} | lr {lr:.2e} | {elapsed:.0f}s')
        t0 = time.time()

    # ── Save checkpoint ────────────────────────────────────
    if it > 0 and it % SAVE_INTERVAL == 0:
        ckpt_path = os.path.join(CKPT_DIR, f'step_{it:06d}.pt')
        torch.save({
            'iter'      : it,
            'model'     : model.state_dict(),
            'optimizer' : optimizer.state_dict(),
            'config'    : cfg,
            'val_loss'  : losses['val'],
        }, ckpt_path)
        # also overwrite latest
        torch.save({
            'iter'      : it,
            'model'     : model.state_dict(),
            'optimizer' : optimizer.state_dict(),
            'config'    : cfg,
            'val_loss'  : losses['val'],
        }, os.path.join(CKPT_DIR, 'latest.pt'))
        print(f'  → Saved checkpoint step_{it:06d}.pt')

    if it == MAX_ITERS:
        break

    # ── Forward + backward ─────────────────────────────────
    x, y = get_batch('train')
    with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
        _, loss = model(x, y)

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

print('\nTraining done!')
with open(os.path.join(OUT_DIR, 'train_log.json'), 'w') as f:
    json.dump(log, f, indent=2)
print('Log saved → train_log.json')

## 8. Quick Inference Check

In [ ]:
model.eval()

prompts = [
    'Urang Sunda',
    'Gunung Gede mangrupa',
    'Bahasa Sunda téh',
]

for prompt in prompts:
    ids = tokenizer.encode(prompt).ids
    x   = torch.tensor([ids], dtype=torch.long, device=device)
    out = model.generate(x, max_new_tokens=50, temperature=0.8, top_k=50)
    generated = tokenizer.decode(out[0].tolist())
    print(f'>> {prompt}')
    print(f'   {generated}')
    print()

## Next

Training done → upload `checkpoints/latest.pt` ke HuggingFace → `05_eval.ipynb`.